In [90]:
import pandas as pd
import numpy as np
import math

In [91]:
num_parte_hijo= pd.read_csv('/content/reglas_numero_parte_hijo.csv')
num_parte_hijo_consignatario=pd.read_csv('/content/reglas_numero_parte_hijo_consigantario.csv')

In [92]:
num_parte_hijo.shape

(671, 5)

In [93]:
num_parte_hijo.head()

,NUMERO_PARTE_HIJO,PIEZAS_POR_PAQ_HIJO,PESO_MAX_DESPACHO_HIJO,max_discos,maximo_peso_neto
0,112111092,NOA,9.50,16,8.780
1,RM2020-189567-ERMA,NOA,NOA,11,19.096
2,68503396SA,NOA,NOA,11,13.860
3,112111098,NOA,9.90,11,9.999
4,S85H0221151X0,NOA,NOA,10,9.810


In [94]:
num_parte_hijo.isna().sum()

,0
NUMERO_PARTE_HIJO,1
PIEZAS_POR_PAQ_HIJO,1
PESO_MAX_DESPACHO_HIJO,4
max_discos,0
maximo_peso_neto,0


In [95]:
num_parte_hijo['PIEZAS_POR_PAQ_HIJO'].value_counts()

,count
PIEZAS_POR_PAQ_HIJO,
NOA,598
1,27
2,23
3,14
400,2
100,2
98,1
500,1
200,1


In [96]:
num_parte_hijo['PESO_MAX_DESPACHO_HIJO'].value_counts()

,count
PESO_MAX_DESPACHO_HIJO,
NOA,553
2.00,18
20.00,16
4.80,10
4.00,8
1.0,5
3.0,4
6.00,4
3.00,4


In [97]:
num_parte_hijo['PIEZAS_POR_PAQ_HIJO']=num_parte_hijo['PIEZAS_POR_PAQ_HIJO'].replace({'NOA':0,np.nan:0})
num_parte_hijo['PIEZAS_POR_PAQ_HIJO']=num_parte_hijo['PIEZAS_POR_PAQ_HIJO'].astype(int)

num_parte_hijo['PESO_MAX_DESPACHO_HIJO']=num_parte_hijo['PESO_MAX_DESPACHO_HIJO'].replace({'NOA':0,np.nan:0})
num_parte_hijo['PESO_MAX_DESPACHO_HIJO']=num_parte_hijo['PESO_MAX_DESPACHO_HIJO'].astype(float)

In [98]:
num_parte_hijo.describe().T

,count,mean,std,min,25%,50%,75%,max
PIEZAS_POR_PAQ_HIJO,671.0,2.898659,30.791429,0.000,0.000,0.000,0.0000,500.000
PESO_MAX_DESPACHO_HIJO,671.0,1.201937,3.780454,0.000,0.000,0.000,0.0000,20.000
max_discos,671.0,2.994039,1.583486,0.000,2.000,3.000,3.0000,16.000
maximo_peso_neto,671.0,8.341490,5.928087,0.522,2.819,7.899,12.2925,24.975


In [99]:
lista_maxmax_discos=[]
lista_maxmax_peso=[]
shape_reglas_optimizadas=[]
for PIEZAS_POR_PAQ_HIJO,max_discos,PESO_MAX_DESPACHO_HIJO,maximo_peso_neto in zip(num_parte_hijo['PIEZAS_POR_PAQ_HIJO'],num_parte_hijo['max_discos'],num_parte_hijo['PESO_MAX_DESPACHO_HIJO'],num_parte_hijo['maximo_peso_neto']):

  if ((PIEZAS_POR_PAQ_HIJO>max_discos) | (PESO_MAX_DESPACHO_HIJO>maximo_peso_neto)):
    if PIEZAS_POR_PAQ_HIJO==0:
      numero_discos=(PESO_MAX_DESPACHO_HIJO*max_discos)/maximo_peso_neto
      numero_discos=math.floor(numero_discos)
      lista_maxmax_discos.append(numero_discos)
      lista_maxmax_peso.append(PESO_MAX_DESPACHO_HIJO)

    elif PESO_MAX_DESPACHO_HIJO==0:
      peso_total=(PIEZAS_POR_PAQ_HIJO*maximo_peso_neto)/max_discos
      peso_total=math.floor(peso_total)
      lista_maxmax_discos.append(PIEZAS_POR_PAQ_HIJO)
      lista_maxmax_peso.append(peso_total)

    else:
      lista_maxmax_discos.append(PIEZAS_POR_PAQ_HIJO)
      lista_maxmax_peso.append(PESO_MAX_DESPACHO_HIJO)


  else:
    lista_maxmax_discos.append(max_discos)
    lista_maxmax_peso.append(maximo_peso_neto)
    shape_reglas_optimizadas.append(max_discos)

num_parte_hijo['Discos_maximos_definidos']=lista_maxmax_discos
num_parte_hijo['Peso_maximo_definido']=lista_maxmax_peso



In [100]:
num_parte_hijo.head()

,NUMERO_PARTE_HIJO,PIEZAS_POR_PAQ_HIJO,PESO_MAX_DESPACHO_HIJO,max_discos,maximo_peso_neto,Discos_maximos_definidos,Peso_maximo_definido
0,112111092,0,9.5,16,8.780,17,9.500
1,RM2020-189567-ERMA,0,0.0,11,19.096,11,19.096
2,68503396SA,0,0.0,11,13.860,11,13.860
3,112111098,0,9.9,11,9.999,11,9.999
4,S85H0221151X0,0,0.0,10,9.810,10,9.810


##Numero de reglas opti


In [101]:
print(f"Numero de reglas optimizadas por numero de parte (NUMERO_PARTE_HIJO) {len(shape_reglas_optimizadas)}, lo que quiere decir que del dataset optimizamos un {round((len(shape_reglas_optimizadas)/len(lista_maxmax_discos))*100,2)}% de las reglas debido a que no reflejaban un límite verdadero (un registro histórico las rompió sin ser desarmado) o no habían reglas definidas (ahora se definieron con el registro máximo historico por ese numero de parte)")

Numero de reglas optimizadas por numero de parte (NUMERO_PARTE_HIJO) 587, lo que quiere decir que del dataset optimizamos un 87.48% de las reglas debido a que no reflejaban un límite verdadero (un registro histórico las rompió sin ser desarmado) o no habían reglas definidas (ahora se definieron con el registro máximo historico por ese numero de parte)


#Num parte hijo por consignatario

In [102]:
num_parte_hijo_consignatario['PIEZAS_POR_PAQ_HIJO']=num_parte_hijo_consignatario['PIEZAS_POR_PAQ_HIJO'].replace({'NOA':0,np.nan:0})
num_parte_hijo_consignatario['PIEZAS_POR_PAQ_HIJO']=num_parte_hijo_consignatario['PIEZAS_POR_PAQ_HIJO'].astype(int)

num_parte_hijo_consignatario['PESO_MAX_DESPACHO_HIJO']=num_parte_hijo_consignatario['PESO_MAX_DESPACHO_HIJO'].replace({'NOA':0,np.nan:0})
num_parte_hijo_consignatario['PESO_MAX_DESPACHO_HIJO']=num_parte_hijo_consignatario['PESO_MAX_DESPACHO_HIJO'].astype(float)

In [103]:
num_parte_hijo_consignatario.describe().T

,count,mean,std,min,25%,50%,75%,max
PIEZAS_POR_PAQ_HIJO,793.0,2.467844,28.339247,0.000,0.000,0.0,0.00,500.000
PESO_MAX_DESPACHO_HIJO,793.0,1.183733,3.832464,0.000,0.000,0.0,0.00,20.000
max_discos,793.0,2.939470,1.532104,1.000,2.000,2.0,3.00,16.000
maximo_peso_neto,793.0,8.318902,5.776453,0.522,2.946,8.0,12.06,24.975


In [104]:
lista_maxmax_discos=[]
lista_maxmax_peso=[]
for PIEZAS_POR_PAQ_HIJO,max_discos,PESO_MAX_DESPACHO_HIJO,maximo_peso_neto in zip(num_parte_hijo_consignatario['PIEZAS_POR_PAQ_HIJO'],num_parte_hijo_consignatario['max_discos'],num_parte_hijo_consignatario['PESO_MAX_DESPACHO_HIJO'],num_parte_hijo_consignatario['maximo_peso_neto']):

  if ((PIEZAS_POR_PAQ_HIJO>max_discos) | (PESO_MAX_DESPACHO_HIJO>maximo_peso_neto)):
    if PIEZAS_POR_PAQ_HIJO==0:
      numero_discos=(PESO_MAX_DESPACHO_HIJO*max_discos)/maximo_peso_neto
      numero_discos=math.floor(numero_discos)
      lista_maxmax_discos.append(numero_discos)
      lista_maxmax_peso.append(PESO_MAX_DESPACHO_HIJO)

    elif PESO_MAX_DESPACHO_HIJO==0:
      peso_total=(PIEZAS_POR_PAQ_HIJO*maximo_peso_neto)/max_discos
      peso_total=math.floor(peso_total)
      lista_maxmax_discos.append(PIEZAS_POR_PAQ_HIJO)
      lista_maxmax_peso.append(peso_total)

    else:
      lista_maxmax_discos.append(PIEZAS_POR_PAQ_HIJO)
      lista_maxmax_peso.append(PESO_MAX_DESPACHO_HIJO)


  else:
    lista_maxmax_discos.append(max_discos)
    lista_maxmax_peso.append(maximo_peso_neto)

num_parte_hijo_consignatario['Discos_maximos_definidos']=lista_maxmax_discos
num_parte_hijo_consignatario['Peso_maximo_definido']=lista_maxmax_peso



In [105]:
num_parte_hijo_consignatario.head()

,CONSIGNATARIO,NUMERO_PARTE_HIJO,PIEZAS_POR_PAQ_HIJO,PESO_MAX_DESPACHO_HIJO,max_discos,maximo_peso_neto,Discos_maximos_definidos,Peso_maximo_definido
0,N000125857,112111092,0,9.5,16,8.780,17,9.500
1,N000125857,112111098,0,9.9,11,9.999,11,9.999
2,N000110074,68503396SA,0,0.0,11,13.860,11,13.860
3,N000116294,RM2020-189567-ERMA,0,0.0,11,19.096,11,19.096
4,N000125857,12113020,0,9.0,10,5.750,15,9.000


In [106]:
from google.colab import files
num_parte_hijo.to_csv('num_parte_hijo_reglas_finaless.csv',encoding='utf-8-sig',index=False)
num_parte_hijo_consignatario.to_csv('num_parte_hijo_consignatario_reglas_finaless.csv',encoding='utf-8-sig',index=False)

In [107]:
files.download('num_parte_hijo_reglas_finaless.csv')
files.download('num_parte_hijo_consignatario_reglas_finaless.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>